# AdaGrad Simulations

## Parameters:

In [0]:
from Solver import AdaGrad, NonlocalSolverAdaGrad
from sklearn.model_selection import ParameterGrid
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import jax 
import jax.numpy as jnp
import os

param_grid = {'lr': [0.1, 0.01]}
n_learning_rates = len(param_grid['lr'])
param_list = list(ParameterGrid(param_grid))

dL = lambda y: 2 * (y - 4) # Derivative of the function to minimize
f = lambda x, y: 0.0 # Rhs of the ODE without the nonlocal part

# Crear carpeta para guardar figuras si no existe
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

## AdaGrad - Discrete

In [0]:
# Create subplots for visualizing the results for different learning rates
fig_theta = make_subplots(rows=1, cols=n_learning_rates, subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']])
fig_accumulated_gradients_result = make_subplots(rows=1, cols=n_learning_rates, subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']])

# Iterate over each learning rate in the parameter grid
for i, lr in enumerate(param_grid['lr']):

    # Filter the list of parameters for the current learning rate
    filtered_params = [p for p in param_list if p['lr'] == lr]

    # Set the number of epochs based on the learning rate
    if lr == 0.1:
        epochs = int(1500)  # 2000 epochs for learning rate 0.1
    elif lr == 0.01:
        epochs = int(150000) # 200,000 epochs for learning rate 0.01

    # Iterate over the filtered parameter configurations
    for params in filtered_params:
        theta_initial = 1.0  # Initial value for theta
        
        print(f'\nAdaGrad Configuration: {params}')
        
        # Initialize and solve the AdaGrad optimization problem
        solver = AdaGrad(dL=dL, lr=lr, epochs=epochs)
        solver.solve(theta_initial=theta_initial)
        
        # Add the theta results to the subplot for theta convergence
        fig_theta.add_trace(go.Scatter(
            x=list(range(epochs)),  # X-axis: epoch numbers
            y=solver.theta_result,  # Y-axis: theta values over epochs
            mode='lines',           # Line plot
            showlegend=False,       # Do not show legend for each individual trace
            legendgroup=f'LR={lr}', # Group traces by learning rate
        ), row=1, col=i+1)  # Add to the appropriate subplot
        
        # Add the accumulated gradient results to the subplot for accumulated gradients
        fig_accumulated_gradients_result.add_trace(go.Scatter(
            x=list(range(epochs)),  # X-axis: epoch numbers
            y=solver.accumulated_gradients_result,  # Y-axis: accumulated gradients over epochs
            mode='markers',          # Scatter plot with markers
            marker=dict(size=3),     # Marker size
            legendgroup=f'LR={lr}',  # Group traces by learning rate
            showlegend=False         # Do not show legend for each individual trace
        ), row=1, col=i+1)  # Add to the appropriate subplot

# Update layout for the theta convergence plot
fig_theta.update_layout(title_text='Theta values convergence trajectories for AdaGrad', showlegend=True)
fig_accumulated_gradients_result.update_layout(title_text='Accumulated gradients convergence trajectories for AdaGrad', showlegend=True)

# Update the axes labels for the theta plot
fig_theta.update_xaxes(title_text="k")
fig_theta.update_yaxes(tickformat=".1f", title_text="Theta_k")
fig_theta.update_layout(
    width=1500,  # Width in pixels
    height=600   # Height in pixels
)

# Update the axes labels for the accumulated gradients plot
fig_accumulated_gradients_result.update_xaxes(title_text="k")
fig_accumulated_gradients_result.update_yaxes(title_text="G_k")
fig_accumulated_gradients_result.update_layout(
    width=1500,  # Width in pixels
    height=600   # Height in pixels
)

# Guardar los gráficos como PNG en la carpeta figures
fig_theta.write_image(os.path.join(figures_dir, "adagrad_theta.png"))
fig_accumulated_gradients_result.write_image(os.path.join(figures_dir, "adagrad_gradients.png"))

print(f"Figuras guardadas como PNG en la carpeta '{figures_dir}'")


## Nonlocal AdaGrad

In [0]:
# Create subplots for visualizing theta values and denominators for different learning rates
fig_theta = make_subplots(rows=1, cols=n_learning_rates, 
                          subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']])
fig_G = make_subplots(rows=1, cols=n_learning_rates, 
                      subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']])

# Update the layout titles for the plots
fig_theta.update_layout(title_text='Theta values convergence trajectory for the first-order nonlocal continuous AdaGrad')
fig_G.update_layout(title_text='G over time for the first-order nonlocal continuous AdaGrad')

# Iterate over each learning rate in the parameter grid
for i, lr in enumerate(param_grid['lr']):

    # Filter the list of parameters for the current learning rate
    filtered_params = [p for p in param_list if p['lr'] == lr]

    # Set the time span based on the learning rate
    if lr == 0.1:
        epochs = int(1500) 
    elif lr == 0.01:
        epochs = int(150000) 

    t = [1e-12, epochs * lr]

    # Iterate over the filtered parameter configurations
    for params in filtered_params:
        print(f'\nNonlocal Continuous AdaGrad Configuration: {params}')

        # Initialize and solve the nonlocal solver using AdaGrad method
        solver = NonlocalSolverAdaGrad(f=f, dL=dL, t_span=t, y0=jnp.array([1.0]), alpha=params['lr'])
        t_values, y_values = solver.solve()
    
        # Add the theta results to the subplot for theta convergence
        fig_theta.add_trace(go.Scatter(
            x=t_values / params['lr'],  # Normalize time by the learning rate
            y=y_values,                 # Theta values over time
            mode='lines',               # Line plot
            showlegend=False,           # Do not show legend for each individual trace
            legendgroup=f'LR={lr}',     # Group traces by learning rate
        ), row=1, col=i+1)  # Add to the appropriate subplot
        
        # Retrieve the denominator values from the solver
        denominators = solver._last_G
        
        # Add the denominators to the subplot for denominators over time
        fig_G.add_trace(go.Scatter(
            x=denominators[:, 0] / params['lr'], 
            y=denominators[:, 1],      
            mode='markers',                                       # Scatter plot with markers
            marker=dict(size=3),                                  # Marker size
            showlegend=False,                                     # Do not show legend for each individual trace
            legendgroup=f'LR={lr}',                               # Group traces by learning rate
        ), row=1, col=i+1)  # Add to the appropriate subplot

# Update axes labels and layout for the theta plot
fig_theta.update_xaxes(title_text="t/alpha")
fig_theta.update_yaxes(tickformat=".1f", title_text="Theta(t)")
fig_theta.update_layout(
    width=1500,  # Width in pixels
    height=600   # Height in pixels
)

# Update axes labels and layout for the denominators plot
fig_G.update_xaxes(title_text="t/alpha")
fig_G.update_yaxes(title_text="G(t)")
fig_G.update_layout(
    width=1500,  # Width in pixels
    height=600   # Height in pixels
)

# Show the figures
fig_theta.write_image(os.path.join(figures_dir, "nonlocal_adagrad_theta.png"))
fig_G.write_image(os.path.join(figures_dir, "nonlocal_g_gradient.png")) 
print(f"Figuras guardadas como PNG en la carpeta '{figures_dir}'")

## Both Models together

In [0]:
# Create a figure for theta trajectories
fig_theta = make_subplots(
    rows=1,
    cols=2,  # Two columns, one for each learning rate
    subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
)

# Create a figure for accumulated gradients/denominators
fig_gradients = make_subplots(
    rows=1,
    cols=2,  # Two columns, one for each learning rate
    subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
)

# Iterate over each learning rate
for i, lr in enumerate(param_grid['lr']):
    # Filter the list of parameters for the current learning rate
    filtered_params_adagrad = [p for p in param_list if p['lr'] == lr]
    filtered_params_nonlocal = [p for p in param_list if p['lr'] == lr]

    # Define epochs and time spans based on the learning rate
    if lr == 0.1:
        epochs = int(1500)
    elif lr == 0.01:
        epochs = int(150000) 

    t = [1e-12, epochs * lr]    

    # AdaGrad simulations
    for params in filtered_params_adagrad:
        theta_initial = 1.0
        print(f'\nAdaGrad Configuration: {params}')
        
        # Solve the optimization problem with AdaGrad
        solver = AdaGrad(dL=dL, lr=lr, epochs=epochs)
        solver.solve(theta_initial=theta_initial)
        
        # Add the theta results from AdaGrad to the theta figure
        fig_theta.add_trace(go.Scatter(
            x=list(range(epochs)),
            y=solver.theta_result,
            mode='markers',
            marker=dict(symbol='x', size=4, line=dict(width=0.01), color='red'),  # Small dots for AdaGrad
            name='AdaGrad',  # Simplified name
            legendgroup='AdaGrad', 
            showlegend=(i == 0)  # Show legend only once
        ), row=1, col=i+1)
        
        # Add the accumulated gradient results from AdaGrad to the gradients figure
        fig_gradients.add_trace(go.Scatter(
            x=list(range(epochs)),
            y=solver.accumulated_gradients_result,
            mode='markers',
            marker=dict(symbol='x', size=4, line=dict(width=0.01), color='red'),  # Small dots for AdaGrad
            name='AdaGrad Gradients',  # Simplified name
            legendgroup='AdaGrad Gradients', 
            showlegend=(i == 0)  # Show legend only once
        ), row=1, col=i+1)

    # Nonlocal AdaGrad simulations
    for params in filtered_params_nonlocal:
        print(f'\nNonlocal Continuous AdaGrad Configuration: {params}')

        # Solve the optimization problem with Nonlocal AdaGrad
        solver_nonlocal = NonlocalSolverAdaGrad(f=f, dL=dL, t_span=t, y0=jnp.array([1.0]), alpha=params['lr'])
        t_values, y_values = solver_nonlocal.solve()

        # Add the theta results from Nonlocal AdaGrad to the theta figure
        fig_theta.add_trace(go.Scatter(
            x=t_values / params['lr'],
            y=y_values,
            mode='lines',  # Line for Nonlocal AdaGrad
            line=dict(color='blue'),
            name='Nonlocal AdaGrad',  # Simplified name
            legendgroup='Nonlocal AdaGrad', 
            showlegend=(i == 0)  # Show legend only once
        ), row=1, col=i+1)

        # Add the denominators to the gradients figure
        denominators = solver_nonlocal._last_G
        fig_gradients.add_trace(go.Scatter(
            x=denominators[:, 0] / params['lr'], 
            y=denominators[:, 1],      
            mode='lines',  # Line for Nonlocal AdaGrad
            line=dict(color='blue'),
            name='Nonlocal AdaGrad Gradients',  # Simplified name
            legendgroup='Nonlocal AdaGrad Gradients', 
            showlegend=(i == 0)  # Show legend only once
        ), row=1, col=i+1)

# Update the layout and axes for the theta figure
fig_theta.update_layout(
    title_text='Theta Convergence Trajectories for Nonlocal AdaGrad',
    width=1200,
    height=600
)
fig_theta.update_xaxes(title_text="t/alpha")
fig_theta.update_yaxes(tickformat=".1f", title_text="Theta Values")

# Update the layout and axes for the gradients figure
fig_gradients.update_layout(
    title_text='Gradients Trajectories for Nonlocal AdaGrad',
    width=1200,
    height=600
)
fig_gradients.update_xaxes(title_text="t/alpha")
fig_gradients.update_yaxes(title_text="Gradients Values")

fig_theta.write_image(os.path.join(figures_dir, "adagrad_vs_nonlocal_theta.png"))
fig_gradients.write_image(os.path.join(figures_dir, "adagrad_vs_nonlocal_g.png"))
print(f"Figuras guardadas como PNG en la carpeta '{figures_dir}'")
